# Task 2: Unsupervised Domain Adaptation (UDA)
**Course:** EE-5102 / CS-6304: Advanced Topics in Machine Learning  
**Assignment:** Programming Assignment 1: Beyond IID and Closed-Set Assumptions  

In this task, we address distribution shift on the **PACS** dataset:
- **Source Domains (Labeled):** Photo (P), Art Painting (A), Cartoon (C) with stratified 80/20 train/val splits (seed `6304`).
- **Target Domain (Unlabeled during adaptation):** Sketch (S).
- **Architecture:** ResNet-18 (`IMAGENET1K_V1`) fine-tuned end-to-end with a 7-class head.
- **BatchNorm Policy:** All BatchNorm running means and variances are frozen at ImageNet values (`bn.eval()`) throughout training; affine parameters ($\gamma, eta$) remain trainable.
- **Methods Compared:**
  1. **Source-only ERM** (reused as the Task 3 ERM baseline)
  2. **DAN** (Multi-kernel RBF MMD discrepancy minimization)
  3. **DANN** (Adversarial domain discriminator with GRL schedule)
  4. **CDAN** (Class-conditional adversarial alignment $g(x) = 	ext{vec}(f \otimes p)$)
- **Protocol:** Strict **Two-Phase Execution**:
  - **Phase A:** Train models and select checkpoints strictly using mean source-validation macro-F1.
  - **Phase B:** Lock checkpoints and evaluate on the Sketch target, computing domain separability and negative transfer.


---
## Step 1: Environment Setup, Global Seed & Hardware Detection


In [1]:
import os
import sys
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns

# Global reproducibility seed
from common.seed import set_seed, SEED
set_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using hardware device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Ensure required workspace directories exist
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('figures/task2', exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('cache', exist_ok=True)


Using hardware device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


---
## Step 2: PACS Dataset Loading & Balanced Batch Sampler
- Sources: Photo, Art Painting, Cartoon (80/20 stratified split, seed 6304).
- Target: Sketch (unlabeled during adaptation).
- Each adaptation update draws 8 Photo + 8 Art + 8 Cartoon (24 source total) and 24 Sketch (24 target total).


In [2]:
from shared.pacs import PACS_CLASSES, PACS_DOMAINS
from shared.pacs_protocol import get_pacs_datasets, BalancedDomainBatchSampler

pacs_data = get_pacs_datasets(pacs_root='data/PACS', seed=SEED)

# Dataloaders for training
batch_size_per_source = 8
batch_size_target = 24

loader_p = DataLoader(pacs_data['source_train']['photo'], batch_size=batch_size_per_source, shuffle=True, drop_last=True)
loader_a = DataLoader(pacs_data['source_train']['art_painting'], batch_size=batch_size_per_source, shuffle=True, drop_last=True)
loader_c = DataLoader(pacs_data['source_train']['cartoon'], batch_size=batch_size_per_source, shuffle=True, drop_last=True)
loader_t = DataLoader(pacs_data['target_adapt'], batch_size=batch_size_target, shuffle=True, drop_last=True)

# Validation dataloaders (source domains)
val_loaders = {
    'Photo': DataLoader(pacs_data['source_val']['photo'], batch_size=32, shuffle=False),
    'Art': DataLoader(pacs_data['source_val']['art_painting'], batch_size=32, shuffle=False),
    'Cartoon': DataLoader(pacs_data['source_val']['cartoon'], batch_size=32, shuffle=False)
}

# Target evaluation loader (Sketch) - Strictly for Phase B!
target_eval_loader = DataLoader(pacs_data['target_eval'], batch_size=32, shuffle=False)

print(f"PACS Loaded successfully!")
for d in ['photo', 'art_painting', 'cartoon']:
    print(f"  Source {d:12s}: Train={len(pacs_data['source_train'][d])}, Val={len(pacs_data['source_val'][d])}")
print(f"  Target Sketch       : Total={len(pacs_data['target_eval'])}")
print(f"  Class Names ({len(PACS_CLASSES)}): {PACS_CLASSES}")


PACS Loaded successfully!
  Source photo       : Train=1336, Val=334
  Source art_painting: Train=1638, Val=410
  Source cartoon     : Train=1875, Val=469
  Target Sketch       : Total=3929
  Class Names (7): ['dog', 'elephant', 'giraffe', 'guitar', 'horse', 'house', 'person']


---
## Step 3: Architecture & BatchNorm Freezing Policy
- ResNet-18 with ImageNet pretrained weights and 7-class linear head.
- **BatchNorm Policy:** All BatchNorm running means and variances are frozen at pretrained ImageNet values (`bn.eval()`). Trainable scale ($\gamma$) and bias ($eta$) remain active.


In [3]:
from task2.models.backbone import freeze_bn_running_stats
from task2.models.classifier_head import PACSResNet18

# Instantiate model
test_model = PACSResNet18(num_classes=7).to(DEVICE)

# Sanity check: verify BatchNorm modules stay in eval mode during train()
test_model.train()
all_bn_eval = all(not m.training for m in test_model.modules() if isinstance(m, nn.BatchNorm2d))
print(f"BatchNorm Freezing Policy Check: All BN modules in eval mode? {all_bn_eval}")
assert all_bn_eval, "BatchNorm modules must remain in eval mode to prevent implicit adaptation!"


BatchNorm Freezing Policy Check: All BN modules in eval mode? True


---
## Step 4: Training Pipeline (Phase A - Source Validation Checkpoint Selection)
Trains for at most 30 source epochs using AdamW ($	ext{lr}=10^{-4}, 	ext{wd}=10^{-4}$).  
Early stopping triggers after 5 epochs without improvement in mean source-validation macro-F1.  
Target labels are strictly masked during training!


In [4]:
from task2.train import train_adaptation


---
## Step 5: Train Source-Only ERM Baseline
Trains ordinary empirical risk minimization over the 3 source domains.  
**Crucial:** We save this checkpoint to `checkpoints/pacs_erm_baseline.pt` so it can be reused unchanged as the Task 3 ERM baseline!


In [5]:
print("=== Training Source-Only ERM Baseline ===")
model_source_only, hist_source_only = train_adaptation(
    method='source_only',
    checkpoint_path='checkpoints/pacs_erm_baseline.pt',
    device=DEVICE
)
print("Source-only model saved to checkpoints/pacs_erm_baseline.pt")


=== Training Source-Only ERM Baseline ===
[11:00:32] INFO: Initializing training for method=source_only on device=cuda
[11:01:14] INFO: [SOURCE_ONLY] Epoch 01/30 | Cls Loss: 0.4835 | Align Loss: 0.0000 | Mean Val F1: 0.8629
[11:01:49] INFO: [SOURCE_ONLY] Epoch 02/30 | Cls Loss: 0.2528 | Align Loss: 0.0000 | Mean Val F1: 0.8974
[11:02:23] INFO: [SOURCE_ONLY] Epoch 03/30 | Cls Loss: 0.1740 | Align Loss: 0.0000 | Mean Val F1: 0.9173
[11:02:56] INFO: [SOURCE_ONLY] Epoch 04/30 | Cls Loss: 0.1392 | Align Loss: 0.0000 | Mean Val F1: 0.9192
[11:03:28] INFO: [SOURCE_ONLY] Epoch 05/30 | Cls Loss: 0.1220 | Align Loss: 0.0000 | Mean Val F1: 0.9130
[11:04:01] INFO: [SOURCE_ONLY] Epoch 06/30 | Cls Loss: 0.1036 | Align Loss: 0.0000 | Mean Val F1: 0.9023
[11:04:35] INFO: [SOURCE_ONLY] Epoch 07/30 | Cls Loss: 0.0894 | Align Loss: 0.0000 | Mean Val F1: 0.9235
[11:05:07] INFO: [SOURCE_ONLY] Epoch 08/30 | Cls Loss: 0.0735 | Align Loss: 0.0000 | Mean Val F1: 0.9052
[11:05:40] INFO: [SOURCE_ONLY] Epoch 09/3

---
## Step 6: Train DAN (MMD Alignment)
Adds the Maximum Mean Discrepancy penalty ($\lambda_{	ext{MMD}} = 1$) between source and target representations.


In [6]:
print("=== Training DAN (MMD Alignment) ===")
model_dan, hist_dan = train_adaptation(
    method='dan',
    lambda_mmd=1.0,
    checkpoint_path='checkpoints/pacs_dan.pt',
    device=DEVICE
)


=== Training DAN (MMD Alignment) ===
[11:08:21] INFO: Initializing training for method=dan on device=cuda
[11:09:05] INFO: [DAN        ] Epoch 01/30 | Cls Loss: 0.4723 | Align Loss: 0.1609 | Mean Val F1: 0.8742
[11:09:48] INFO: [DAN        ] Epoch 02/30 | Cls Loss: 0.2499 | Align Loss: 0.1362 | Mean Val F1: 0.9104
[11:10:40] INFO: [DAN        ] Epoch 03/30 | Cls Loss: 0.1889 | Align Loss: 0.1310 | Mean Val F1: 0.9159
[11:11:34] INFO: [DAN        ] Epoch 04/30 | Cls Loss: 0.1455 | Align Loss: 0.1322 | Mean Val F1: 0.9255
[11:12:32] INFO: [DAN        ] Epoch 05/30 | Cls Loss: 0.1358 | Align Loss: 0.1200 | Mean Val F1: 0.9161
[11:13:30] INFO: [DAN        ] Epoch 06/30 | Cls Loss: 0.1087 | Align Loss: 0.1214 | Mean Val F1: 0.9143
[11:14:25] INFO: [DAN        ] Epoch 07/30 | Cls Loss: 0.1093 | Align Loss: 0.1276 | Mean Val F1: 0.9264
[11:15:23] INFO: [DAN        ] Epoch 08/30 | Cls Loss: 0.0835 | Align Loss: 0.1154 | Mean Val F1: 0.9258
[11:16:22] INFO: [DAN        ] Epoch 09/30 | Cls Loss:

---
## Step 7: Train DANN (Adversarial Alignment)
Attaches a binary domain discriminator to the 512-d feature with a gradient reversal layer.


In [7]:
print("=== Training DANN (Adversarial Alignment) ===")
model_dann, hist_dann = train_adaptation(
    method='dann',
    checkpoint_path='checkpoints/pacs_dann.pt',
    device=DEVICE
)


=== Training DANN (Adversarial Alignment) ===
[11:22:12] INFO: Initializing training for method=dann on device=cuda
[11:23:09] INFO: [DANN       ] Epoch 01/30 | Cls Loss: 0.4932 | Align Loss: 0.7011 | Mean Val F1: 0.9129
[11:24:00] INFO: [DANN       ] Epoch 02/30 | Cls Loss: 0.2416 | Align Loss: 0.7997 | Mean Val F1: 0.9154
[11:24:51] INFO: [DANN       ] Epoch 03/30 | Cls Loss: 0.2074 | Align Loss: 0.7945 | Mean Val F1: 0.9070
[11:25:41] INFO: [DANN       ] Epoch 04/30 | Cls Loss: 0.1499 | Align Loss: 0.7575 | Mean Val F1: 0.8404
[11:26:28] INFO: [DANN       ] Epoch 05/30 | Cls Loss: 0.1296 | Align Loss: 0.7586 | Mean Val F1: 0.9181
[11:27:10] INFO: [DANN       ] Epoch 06/30 | Cls Loss: 0.0951 | Align Loss: 0.7624 | Mean Val F1: 0.9068
[11:27:52] INFO: [DANN       ] Epoch 07/30 | Cls Loss: 0.0960 | Align Loss: 0.7314 | Mean Val F1: 0.8916
[11:28:42] INFO: [DANN       ] Epoch 08/30 | Cls Loss: 0.0781 | Align Loss: 0.7352 | Mean Val F1: 0.9174
[11:29:35] INFO: [DANN       ] Epoch 09/30 |

---
## Step 8: Train CDAN (Class-Conditional Alignment)
Conditions the domain discriminator on the multilinear feature $g(x) = 	ext{vec}(f \otimes p)$.


In [8]:
print("=== Training CDAN (Class-Conditional Alignment) ===")
model_cdan, hist_cdan = train_adaptation(
    method='cdan',
    checkpoint_path='checkpoints/pacs_cdan.pt',
    device=DEVICE
)


=== Training CDAN (Class-Conditional Alignment) ===
[11:36:58] INFO: Initializing training for method=cdan on device=cuda
[11:37:42] INFO: [CDAN       ] Epoch 01/30 | Cls Loss: 0.4849 | Align Loss: 0.6217 | Mean Val F1: 0.9034
[11:38:36] INFO: [CDAN       ] Epoch 02/30 | Cls Loss: 0.2566 | Align Loss: 0.6981 | Mean Val F1: 0.8391
[11:39:29] INFO: [CDAN       ] Epoch 03/30 | Cls Loss: 0.2056 | Align Loss: 0.7066 | Mean Val F1: 0.8881
[11:40:20] INFO: [CDAN       ] Epoch 04/30 | Cls Loss: 0.1812 | Align Loss: 0.7359 | Mean Val F1: 0.9118
[11:41:06] INFO: [CDAN       ] Epoch 05/30 | Cls Loss: 0.1545 | Align Loss: 0.6961 | Mean Val F1: 0.8949
[11:41:56] INFO: [CDAN       ] Epoch 06/30 | Cls Loss: 0.1347 | Align Loss: 0.7086 | Mean Val F1: 0.8902
[11:42:51] INFO: [CDAN       ] Epoch 07/30 | Cls Loss: 0.1456 | Align Loss: 0.7180 | Mean Val F1: 0.9170
[11:43:44] INFO: [CDAN       ] Epoch 08/30 | Cls Loss: 0.1031 | Align Loss: 0.6910 | Mean Val F1: 0.9226
[11:44:34] INFO: [CDAN       ] Epoch 0

---
## Step 9: Controlled Design Study: DAN Alignment Weight Sweep
Evaluates the effect of varying $\lambda_{	ext{MMD}} \in \{0.1, 10\}$ (nominal $\lambda=1.0$ trained in Step 6).


In [9]:
print("=== Controlled Study: Training DAN with lambda=0.1 ===")
model_dan_low, hist_dan_low = train_adaptation(
    method='dan',
    lambda_mmd=0.1,
    checkpoint_path='checkpoints/pacs_dan_lambda0.1.pt',
    device=DEVICE
)

print("\n=== Controlled Study: Training DAN with lambda=10.0 ===")
model_dan_high, hist_dan_high = train_adaptation(
    method='dan',
    lambda_mmd=10.0,
    checkpoint_path='checkpoints/pacs_dan_lambda10.0.pt',
    device=DEVICE
)


=== Controlled Study: Training DAN with lambda=0.1 ===
[11:54:02] INFO: Initializing training for method=dan on device=cuda
[11:54:45] INFO: [DAN        ] Epoch 01/30 | Cls Loss: 0.4938 | Align Loss: 0.0251 | Mean Val F1: 0.8432
[11:55:28] INFO: [DAN        ] Epoch 02/30 | Cls Loss: 0.2386 | Align Loss: 0.0174 | Mean Val F1: 0.8821
[11:56:11] INFO: [DAN        ] Epoch 03/30 | Cls Loss: 0.1772 | Align Loss: 0.0158 | Mean Val F1: 0.9341
[11:56:54] INFO: [DAN        ] Epoch 04/30 | Cls Loss: 0.1369 | Align Loss: 0.0153 | Mean Val F1: 0.9336
[11:57:37] INFO: [DAN        ] Epoch 05/30 | Cls Loss: 0.1209 | Align Loss: 0.0135 | Mean Val F1: 0.9443
[11:58:33] INFO: [DAN        ] Epoch 06/30 | Cls Loss: 0.0965 | Align Loss: 0.0134 | Mean Val F1: 0.9368
[11:59:30] INFO: [DAN        ] Epoch 07/30 | Cls Loss: 0.1089 | Align Loss: 0.0136 | Mean Val F1: 0.9394
[12:00:27] INFO: [DAN        ] Epoch 08/30 | Cls Loss: 0.0829 | Align Loss: 0.0132 | Mean Val F1: 0.9057
[12:01:12] INFO: [DAN        ] Epoch

---
## Step 10: Plot Training & Alignment Loss Dynamics (Figure 1)
Exports classification loss and alignment/discriminator loss curves to `figures/task2/`.


In [10]:
from common.plotting import plot_training_curves

training_histories = {
    'Source-only': hist_source_only,
    'DAN': hist_dan,
    'DANN': hist_dann,
    'CDAN': hist_cdan
}

# 1. Classification Loss Curves
cls_curves = {m: h['cls_loss'] for m, h in training_histories.items() if 'cls_loss' in h}
plot_training_curves(
    cls_curves,
    title="Source Classification Loss Dynamics",
    xlabel="Epoch",
    ylabel="Classification Loss (Cross-Entropy)",
    save_path="figures/task2/training_loss_curves.png"
)

# 2. Alignment Loss Curves
align_curves = {
    'DAN (MMD)': hist_dan['align_loss'],
    'DANN (Adversarial)': hist_dann['align_loss'],
    'CDAN (Conditional)': hist_cdan['align_loss']
}
plot_training_curves(
    align_curves,
    title="Domain Alignment Objective Dynamics",
    xlabel="Epoch",
    ylabel="Alignment / Discriminator Loss",
    save_path="figures/task2/alignment_loss_curves.png"
)
print("Loss curves saved: figures/task2/training_loss_curves.png and alignment_loss_curves.png")


Loss curves saved: figures/task2/training_loss_curves.png and alignment_loss_curves.png


---
# PHASE B: Transductive Target Evaluation & Alignment Diagnostics
All models, hyperparameter configurations, and checkpoints are now locked.  
Target (Sketch) labels are accessed solely in the cells below for final evaluation.


---
## Step 11: Benchmark Evaluation on PACS & Domain Separability Diagnostic
Runs transductive evaluation on Sketch, computes domain separability scores, and analyzes per-class negative transfer.


In [11]:
from task2.evaluate_final import evaluate_all_adaptation_models

task2_results = evaluate_all_adaptation_models(
    checkpoints={
        'Source-only (ERM)': 'checkpoints/pacs_erm_baseline.pt',
        'DAN': 'checkpoints/pacs_dan.pt',
        'DANN': 'checkpoints/pacs_dann.pt',
        'CDAN': 'checkpoints/pacs_cdan.pt'
    },
    results_path='results/task2_results.json',
    figures_dir='figures/task2',
    device=DEVICE
)

print("\n=== Main Benchmark Results (PACS -> Sketch) ===")
print(f"{'Method':20s} | {'Mean Src Acc':12s} | {'Mean Src F1':11s} | {'Sketch Acc':10s} | {'Sketch F1':9s} | {'Gain':7s} | {'Dom Sep':7s}")
print("-" * 90)
for m_name, res in task2_results['main_results'].items():
    print(f"{m_name:20s} | {res['mean_src_acc']:10.2f}% | {res['mean_src_f1']:11.4f} | {res['target_acc']:8.2f}% | {res['target_f1']:9.4f} | {res['target_gain']:+6.2f}% | {res['domain_separability']:5.2f}%")


[12:11:35] INFO: Evaluating Source-only (ERM) from checkpoints/pacs_erm_baseline.pt...
[12:11:51] INFO: Evaluating DAN from checkpoints/pacs_dan.pt...
[12:12:08] INFO: Evaluating DANN from checkpoints/pacs_dann.pt...


c:\Users\Abdul\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


[12:12:23] INFO: Evaluating CDAN from checkpoints/pacs_cdan.pt...
[12:12:38] INFO: Evaluation complete. Results saved to results/task2_results.json

=== Main Benchmark Results (PACS -> Sketch) ===
Method               | Mean Src Acc | Mean Src F1 | Sketch Acc | Sketch F1 | Gain    | Dom Sep
------------------------------------------------------------------------------------------
Source-only (ERM)    |      93.80% |      0.9373 |    72.74% |    0.7523 |  +0.00% | 99.59%
DAN                  |      94.35% |      0.9420 |    63.86% |    0.5605 |  -8.88% | 90.93%
DANN                 |      94.09% |      0.9410 |    19.65% |    0.0469 | -53.09% | 100.00%
CDAN                 |      93.46% |      0.9336 |     2.04% |    0.0057 | -70.71% | 100.00%


---
## Step 12: Controlled Study Table
Evaluates $\lambda_{	ext{MMD}} \in \{0.1, 1.0, 10.0\}$.


In [12]:
from task2.evaluation.metrics import evaluate_model
from task2.evaluation.domain_separability import compute_domain_separability

study_ckpts = {
    'DAN (lambda=0.1)': 'checkpoints/pacs_dan_lambda0.1.pt',
    'DAN (lambda=1.0)': 'checkpoints/pacs_dan.pt',
    'DAN (lambda=10.0)': 'checkpoints/pacs_dan_lambda10.0.pt'
}

print(f"{'Configuration':20s} | {'Mean Src Acc':12s} | {'Mean Src F1':11s} | {'Dom Sep':8s} | {'Sketch Acc':10s} | {'Sketch F1':9s}")
print("-" * 80)

for s_name, s_ckpt in study_ckpts.items():
    if os.path.exists(s_ckpt):
        model = PACSResNet18(num_classes=7).to(DEVICE)
        model.load_state_dict(torch.load(s_ckpt, map_location=DEVICE))
        model.eval()
        
        # Source val
        src_accs, src_f1s, src_feats_list = [], [], []
        for _, v_loader in val_loaders.items():
            _, f1, acc, _, _, _, feats = evaluate_model(model, v_loader, DEVICE)
            src_accs.append(acc)
            src_f1s.append(f1)
            src_feats_list.append(feats)
        src_feats = np.concatenate(src_feats_list, axis=0)
        
        # Target Sketch
        _, tgt_f1, tgt_acc, _, _, _, tgt_feats = evaluate_model(model, target_eval_loader, DEVICE)
        dom_sep = compute_domain_separability(src_feats, tgt_feats, seed=SEED)
        
        print(f"{s_name:20s} | {np.mean(src_accs):10.2f}% | {np.mean(src_f1s):11.4f} | {dom_sep:6.2f}% | {tgt_acc:8.2f}% | {tgt_f1:9.4f}")


Configuration        | Mean Src Acc | Mean Src F1 | Dom Sep  | Sketch Acc | Sketch F1
--------------------------------------------------------------------------------
DAN (lambda=0.1)     |      94.41% |      0.9443 |  96.84% |    74.70% |    0.7311


c:\Users\Abdul\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


DAN (lambda=1.0)     |      94.35% |      0.9420 |  90.93% |    63.86% |    0.5605
DAN (lambda=10.0)    |      88.74% |      0.8860 |  92.17% |    62.71% |    0.5857


c:\Users\Abdul\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
